In [14]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn

# ==========================================
# 1. PREPARACIÓN DE DATOS Y MODELO
# ==========================================
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2023, 0.1994, 0.201])
])

print("Descargando/Cargando datasets...")
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
# Usamos el testloader para la evaluación del ataque
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False, num_workers=2)

print("Cargando el modelo cifar10_resnet20...")
model = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_resnet20", pretrained=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("-" * 30)
print(f"✅ Dispositivo configurado: {device}")
print(f"✅ Tamaño del dataset de test: {len(testset)} imágenes")
print("-" * 30)

Descargando/Cargando datasets...
Cargando el modelo cifar10_resnet20...
------------------------------
✅ Dispositivo configurado: cpu
✅ Tamaño del dataset de test: 10000 imágenes
------------------------------


Using cache found in /home/anton/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


In [ ]:
def fgsm_attack(images, epsilon, data_grad):
    # 1. Obtener el signo de los gradientes
    sign_data_grad = data_grad.sign()
    
    # 2. Crear la imagen adversaria añadiendo la perturbación
    perturbed_images = images + epsilon * sign_data_grad
    
    # Nota: Debido a la normalización estricta de CIFAR-10, omitimos el "clipping" a [0,1] 
    # en el espacio normalizado, ya que alteraría los rangos estandarizados reales.
    return perturbed_images

def evaluate_evasion_attack(model, testloader, epsilon, min_images=1000):
    print(f"Iniciando evaluación de FGSM (epsilon = {epsilon})...")
    
    # El modelo debe estar en modo evaluación
    model.eval()
    criterion = nn.CrossEntropyLoss()
    
    correct_clean = 0
    correct_adv = 0
    total = 0
    
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        
        # --- A. PRECISION ORIGINAL (Limpia) ---
        # No necesitamos gradientes para evaluar las imágenes originales
        with torch.no_grad():
            outputs_clean = model(images)
            _, predicted_clean = torch.max(outputs_clean.data, 1)
            correct_clean += (predicted_clean == labels).sum().item()
            
        # --- B. ATAQUE ADVERSARIO ---
        # Habilitamos el cálculo de gradientes para las imágenes de entrada
        images.requires_grad = True
        
        # Forward pass para obtener gradientes
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Limpiar gradientes previos del modelo
        model.zero_grad()
        
        # Backward pass para calcular los gradientes de la imagen
        loss.backward()
        
        # Extraer los gradientes de las imágenes
        data_grad = images.grad.data
        
        # Generar las imágenes adversarias
        perturbed_images = fgsm_attack(images, epsilon, data_grad)
        
        # --- C. PRECISION ADVERSARIA ---
        # Evaluamos el modelo de nuevo usando las imágenes manipuladas
        with torch.no_grad():
            outputs_adv = model(perturbed_images)
            _, predicted_adv = torch.max(outputs_adv.data, 1)
            correct_adv += (predicted_adv == labels).sum().item()
            
        total += labels.size(0)
        
        # Detener la evaluación si ya superamos el mínimo requerido (1000)
        if total >= min_images:
            break
        # Calcular métricas finales
    clean_acc = 100 * correct_clean / total
    adv_acc = 100 * correct_adv / total
    
    print("=" * 30)
    print(f"📊 Resultados sobre {total} imágenes:")
    print(f"Precisión Limpia (Sin ataque): {clean_acc:.2f}%")
    print(f"Precisión Adversa (Con FGSM):  {adv_acc:.2f}%")
    print(f"Caída de precisión:            {clean_acc - adv_acc:.2f}%")
    print("=" * 30)

epsilon_value = 0.05
evaluate_evasion_attack(model, testloader, epsilon=epsilon_value, min_images=1000)

Iniciando evaluación de FGSM (epsilon = 1)...
📊 Resultados sobre 1024 imágenes:
Precisión Limpia (Sin ataque): 92.68%
Precisión Adversa (Con FGSM):  9.38%
Caída de precisión:            83.30%
